# Non-Gaussianity of the Reionization kSZ Signal

**Motivation:** The reionization kSZ signal originates from ionized bubbles and is expected to be
highly non-Gaussian. Late-time (post-reionization) kSZ from linear density/velocity fields should
be much closer to Gaussian. Current state-of-the-art separation methods exploit differences in
the **trispectrum** (4-point function). If reionization kSZ has large *beyond*-4-point statistics,
ML methods could potentially outperform trispectrum-based approaches.

**This notebook investigates:**
1. Non-Gaussianity of 21cmFAST kSZ maps (skewness, kurtosis, Minkowski functionals, pixel PDF)
2. **EoR lightcone kSZ** (z~5–12) vs **late-time lightcone kSZ** (z~0.5–5): both are full LOS-integrated signals
3. **Coeval vs lightcone** — does the integrated lightcone wash out or preserve non-Gaussianity?
4. **Simple kSZ (our method) vs wrapper.py kSZ** — the `ksz_reconstruction` wrapper implements
   cumulative optical depth damping exp(-τ), ray-tracing, and box rotation to avoid repetition
   artifacts. How do the maps and their non-Gaussianity compare?
5. ML separability: can a CNN distinguish reionization from late-time kSZ using beyond-2-point info?

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.fft as fft
from scipy import stats
from astropy import units as un
from astropy import constants
from astropy.cosmology import Planck18
from powerbox import get_power
import sys

# 21cmFAST
import py21cmfast as p21c

# Add ksz_reconstruction to path for wrapper comparison
sys.path.insert(0, "/Users/jelte/Documents/PhD/ksz_reconstruction")

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## Physical constants and kSZ functions

Reuse the same kSZ pipeline from notebook 01 (simple coeval projection), plus define
non-Gaussianity diagnostics.

In [2]:
# ── Physical constants ──
_SIGMA_T   = 6.6524587158e-25   # Thomson cross-section [cm²]
_C_KM_S    = 2.99792458e5       # speed of light [km/s]
_T_CMB     = 2.725e6            # CMB temperature [µK]
_M_P       = 1.6726219e-24      # proton mass [g]
_MPC_TO_CM = 3.085677581e24     # Mpc → cm
_MPC_TO_KM = 3.085677581e19    # Mpc → km

# ── Cosmological parameters (Planck 2018) ──
H_PARAM  = 0.6736
OMEGA_M  = 0.315
OMEGA_B  = 0.049


def compute_ksz_map(vz, xHI, density, z, box_len, h=H_PARAM, omega_b=OMEGA_B,
                    axis=2):
    """Project kSZ signal along LOS (simple coeval sum — our method).
    
    kSZ = prefactor * sum(x_e * (1+delta) * v_z) along LOS
    
    This is the simplified version that does NOT include:
    - Cumulative optical depth damping exp(-tau)
    - Ray-tracing / angular diameter distance corrections
    - Box rotation to avoid repetition artifacts
    
    Returns: (N,N) kSZ map in µK.
    """
    n_cells = density.shape[axis]
    Y_He = 0.24
    X_H = 1.0 - Y_He
    rho_crit_0 = 1.8788e-29 * h**2
    rho_b_0 = omega_b * rho_crit_0
    n_e_0 = rho_b_0 / _M_P * (1.0 + X_H) / 2.0
    n_e_z = n_e_0 * (1.0 + z)**3
    dl_cm = (box_len / (1.0 + z)) / n_cells * _MPC_TO_CM
    pref = -_T_CMB * _SIGMA_T * n_e_z * dl_cm / _C_KM_S

    x_e = 1.0 - np.clip(xHI, 0.0, 1.0)
    return (pref * np.sum(x_e * (1.0 + density) * vz, axis=axis)).astype(np.float32)


def compute_ksz_map_lightcone_simple(velocity, xHI, density, redshifts, box_len,
                                      h=H_PARAM, omega_b=OMEGA_B):
    """Our simple kSZ projection applied to a lightcone (per-slice redshift weighting).
    
    Same physics as compute_ksz_map but each z-slice uses its own redshift for
    the prefactor: pref(z) = -T_CMB * sigma_T * n_e(z) * dl(z) / c.
    
    No cumulative optical depth, no ray-tracing, no box rotation.
    """
    n_xy = density.shape[0]
    n_z = density.shape[2]
    Y_He = 0.24
    X_H = 1.0 - Y_He
    rho_crit_0 = 1.8788e-29 * h**2
    rho_b_0 = omega_b * rho_crit_0
    n_e_0 = rho_b_0 / _M_P * (1.0 + X_H) / 2.0
    
    ksz_map = np.zeros((n_xy, n_xy), dtype=np.float64)
    
    for iz in range(n_z):
        z = redshifts[iz]
        a = 1.0 / (1.0 + z)
        n_e_z = n_e_0 * (1.0 + z)**3
        dl_cm = (box_len / (1.0 + z)) / n_z * _MPC_TO_CM
        pref = -_T_CMB * _SIGMA_T * n_e_z * dl_cm / _C_KM_S
        
        x_e = 1.0 - np.clip(xHI[:, :, iz], 0.0, 1.0)
        vz_phys = a * velocity[:, :, iz] * _MPC_TO_KM
        
        ksz_map += pref * x_e * (1.0 + density[:, :, iz]) * vz_phys
    
    return ksz_map.astype(np.float32)


print("kSZ projection functions defined (coeval + lightcone).")

kSZ projection functions defined (coeval + lightcone).


In [3]:
# ── Non-Gaussianity diagnostics ──

def pixel_statistics(map_2d, label=""):
    """Compute pixel-level non-Gaussianity statistics for a 2D map."""
    flat = map_2d.ravel().astype(np.float64)
    flat = flat - flat.mean()
    sigma = flat.std()
    
    skew = stats.skew(flat)
    kurt = stats.kurtosis(flat)  # excess kurtosis (Gaussian = 0)
    
    # Higher-order standardized moments (5th and 6th)
    m5 = np.mean((flat / sigma)**5)
    m6 = np.mean((flat / sigma)**6) - 15  # excess (Gaussian 6th moment = 15)
    
    # Shapiro-Wilk test (on subsample if too large)
    n_sw = min(5000, len(flat))
    _, p_shapiro = stats.shapiro(np.random.choice(flat, n_sw, replace=False))
    
    # D'Agostino-Pearson omnibus test
    _, p_dagostino = stats.normaltest(flat)
    
    result = {
        "label": label,
        "std": sigma,
        "skewness": skew,
        "excess_kurtosis": kurt,
        "m5 (5th moment)": m5,
        "m6_excess (6th moment - 15)": m6,
        "p_shapiro": p_shapiro,
        "p_dagostino": p_dagostino,
    }
    
    print(f"── {label} ──")
    print(f"  std         = {sigma:.4e}")
    print(f"  skewness    = {skew:.4f}   (Gaussian = 0)")
    print(f"  ex. kurtosis= {kurt:.4f}   (Gaussian = 0)")
    print(f"  m5          = {m5:.4f}   (Gaussian = 0)")
    print(f"  m6 excess   = {m6:.4f}   (Gaussian = 0)")
    print(f"  Shapiro p   = {p_shapiro:.2e}  (p < 0.05 → reject Gaussian)")
    print(f"  D'Agostino p= {p_dagostino:.2e}")
    return result


def minkowski_functionals_2d(map_2d, n_thresholds=50):
    """Compute 2D Minkowski functionals V0 (area), V1 (perimeter), V2 (Euler char.)
    as a function of threshold, for a 2D map.
    
    These are sensitive to non-Gaussianity: for Gaussian random fields,
    the MFs have known analytic forms. Deviations indicate non-Gaussianity.
    """
    flat = map_2d.astype(np.float64)
    mu, sigma = flat.mean(), flat.std()
    # Normalized thresholds from -3σ to +3σ
    nu = np.linspace(-3, 3, n_thresholds)
    thresholds = mu + nu * sigma
    
    V0 = np.zeros(n_thresholds)  # fraction of area above threshold
    V1 = np.zeros(n_thresholds)  # boundary length (perimeter)
    V2 = np.zeros(n_thresholds)  # Euler characteristic
    
    ny, nx = map_2d.shape
    
    for i, thr in enumerate(thresholds):
        binary = (map_2d > thr).astype(np.int8)
        
        # V0: area fraction
        V0[i] = binary.mean()
        
        # V1: perimeter — count edges between 0 and 1
        edges_h = np.abs(np.diff(binary, axis=1)).sum()
        edges_v = np.abs(np.diff(binary, axis=0)).sum()
        V1[i] = (edges_h + edges_v) / (nx * ny)
        
        # V2: Euler characteristic via 2x2 pixel patterns
        # χ = (n_vertices - n_edges + n_faces) for the excursion set
        # Quick approximation: count connected components minus holes
        # Using the formula: V2 ≈ (C_4 - C_2) where C_n counts n-corner configs
        c1 = binary[:-1, :-1]
        c2 = binary[:-1, 1:]
        c3 = binary[1:, :-1]
        c4 = binary[1:, 1:]
        s = c1 + c2 + c3 + c4
        # Euler char from 2x2 blocks: +1 for single pixel, -1 for diagonal pair
        n1 = np.sum(s == 1)
        n3 = np.sum(s == 3)
        diag_a = np.sum((c1 == 1) & (c4 == 1) & (c2 == 0) & (c3 == 0))
        diag_b = np.sum((c2 == 1) & (c3 == 1) & (c1 == 0) & (c4 == 0))
        V2[i] = (n1 - n3 + 2 * (diag_a + diag_b)) / ((nx - 1) * (ny - 1))
    
    # Analytic Gaussian predictions
    V0_gauss = 0.5 * (1 - stats.norm.cdf(nu) - stats.norm.cdf(-nu))
    # More precisely: V0 = 1 - Phi(nu)
    V0_gauss = 1 - stats.norm.cdf(nu)
    V1_gauss = (1 / (2 * np.pi)) * np.exp(-nu**2 / 2)  # up to normalization
    V2_gauss = (1 / (2 * np.pi)**1.5) * nu * np.exp(-nu**2 / 2)
    
    return nu, V0, V1, V2, V0_gauss, V1_gauss, V2_gauss


def plot_pixel_pdf(maps_dict, nbins=100):
    """Plot pixel value PDFs for multiple maps, compared to Gaussian."""
    fig, ax = plt.subplots(figsize=(8, 5))
    
    for label, m in maps_dict.items():
        flat = m.ravel().astype(np.float64)
        flat = (flat - flat.mean()) / flat.std()  # standardize
        ax.hist(flat, bins=nbins, density=True, alpha=0.5, label=label, histtype="stepfilled")
    
    x = np.linspace(-5, 5, 300)
    ax.plot(x, stats.norm.pdf(x), "k--", lw=2, label="Gaussian")
    ax.set_xlabel(r"$(T - \bar{T}) / \sigma_T$")
    ax.set_ylabel("PDF")
    ax.set_yscale("log")
    ax.set_ylim(1e-4, 2)
    ax.set_xlim(-5, 5)
    ax.legend()
    ax.set_title("Pixel PDF — non-Gaussianity comparison")
    plt.tight_layout()
    return fig


print("Non-Gaussianity diagnostic functions defined.")

Non-Gaussianity diagnostic functions defined.


## Part 1: Generate kSZ maps — EoR vs late-time lightcones

We generate two lightcones with proper LOS integration:
- **EoR kSZ lightcone** (z=5–12): captures patchy reionization with non-Gaussian bubble structure
- **Late-time kSZ lightcone** (z=0.5–5): post-reionization, fully ionized, kSZ from density/velocity only

We also keep a single **coeval at z=8** to compare coeval vs lightcone non-Gaussianity.

Both lightcones use the **wrapper.py method** (`_Proj_array`) with cumulative exp(-τ),
since both need proper LOS integration. We also compute the EoR lightcone with our
simple method (no exp(-τ)) to compare the two approaches.

In [4]:
# Simulation parameters — kept small to avoid OOM
BOX_LEN = 300    # comoving Mpc
DIM = 256        # high-res grid (reduced from 512)
HII_DIM = 128    # ionization grid (reduced from 256)
SEED = 42

# Redshift ranges
Z_EOR_MIN = 6.0    # EoR lightcone: z=6 to 10 (narrower to save memory)
Z_EOR_MAX = 10.0
Z_LATE_MIN = 1.0   # Late-time lightcone: z=1 to 5
Z_LATE_MAX = 5.0
Z_COEVAL = 8.0     # Single coeval for comparison

resolution = BOX_LEN / HII_DIM * un.Mpc  # cell size for lightconer

from py21cmfast.wrapper.inputs import get_logspaced_redshifts

# Coarser redshift stepping (1.08 instead of 1.04) → fewer nodes → less memory
node_redshifts_eor = get_logspaced_redshifts(
    min_redshift=Z_EOR_MIN,
    max_redshift=Z_EOR_MAX + 2,
    z_step_factor=1.08,
)
node_redshifts_late = get_logspaced_redshifts(
    min_redshift=Z_LATE_MIN,
    max_redshift=Z_LATE_MAX + 2,
    z_step_factor=1.08,
)

print(f"EoR node redshifts: {len(node_redshifts_eor)} steps")
print(f"Late node redshifts: {len(node_redshifts_late)} steps")

inputs_eor = p21c.InputParameters(
    simulation_options=p21c.SimulationOptions(
        BOX_LEN=BOX_LEN, DIM=DIM, HII_DIM=HII_DIM,
    ),
    random_seed=SEED,
    node_redshifts=node_redshifts_eor,
)

inputs_late = p21c.InputParameters(
    simulation_options=p21c.SimulationOptions(
        BOX_LEN=BOX_LEN, DIM=DIM, HII_DIM=HII_DIM,
    ),
    random_seed=SEED,
    node_redshifts=node_redshifts_late,
)

inputs = p21c.InputParameters(
    simulation_options=p21c.SimulationOptions(
        BOX_LEN=BOX_LEN, DIM=DIM, HII_DIM=HII_DIM,
    ),
    random_seed=SEED,
)

# ── Single coeval at z=8 ──
print("\nRunning coeval at z=8...")
coeval_reion = p21c.run_coeval(inputs=inputs, out_redshifts=[Z_COEVAL])[0]
print(f"  xHI mean = {coeval_reion.ionized_box.get('neutral_fraction').mean():.3f}")
print("Done.")

EoR node redshifts: 10 steps
Late node redshifts: 20 steps

Running coeval at z=8...


/Users/jelte/Documents/PhD/noise_analysis/.venv/lib/python3.11/site-packages/attr/_make.py:3323: UserWarning: Resolution is likely too low for accurate evolved density fields. It is recommended that you either increase the resolution (DIM/BOX_LEN) or set the EVOLVE_DENSITY_LINEARLY flag to True. Got DIM=256, BOX_LEN=300.0, resolution=1.171875 Mpc Mpc.
  v(inst, attr, value)


  xHI mean = 0.586
Done.


In [5]:
# ── Coeval kSZ map at z=8 (our simple method) ──
a_coeval = 1.0 / (1.0 + Z_COEVAL)
density_coeval = coeval_reion.density
xHI_coeval = coeval_reion.ionized_box.get("neutral_fraction")
vz_coeval = a_coeval * coeval_reion.velocity_z * _MPC_TO_KM  # physical km/s

ksz_coeval = compute_ksz_map(vz_coeval, xHI_coeval, density_coeval, Z_COEVAL, BOX_LEN)
print(f"Coeval kSZ (z={Z_COEVAL}): std = {ksz_coeval.std():.4f} µK, "
      f"mean xHI = {xHI_coeval.mean():.3f}")

Coeval kSZ (z=8.0): std = 0.8506 µK, mean xHI = 0.586


## Part 2: Generate EoR and late-time lightcones

Run two lightcones using the new py21cmfast API:
- **EoR**: `RectilinearLightconer.between_redshifts(5, max_z)` — captures patchy reionization
- **Late-time**: `RectilinearLightconer.between_redshifts(0.5, 5)` — fully ionized universe

In [ ]:
# ── EoR lightcone (z=6 to 10) ──
from py21cmfast.io.caching import CacheConfig

lc_quantities = ("neutral_fraction", "density", "velocity_z")

lightconer_eor = p21c.RectilinearLightconer.between_redshifts(
    min_redshift=Z_EOR_MIN,
    max_redshift=Z_EOR_MAX,
    resolution=resolution,
    quantities=lc_quantities,
)

print(f"EoR lightconer: z = {lightconer_eor.lc_redshifts.min():.2f} – {lightconer_eor.lc_redshifts.max():.2f}")
print(f"  N slices = {len(lightconer_eor.lc_redshifts)}")

print("\nRunning EoR lightcone...")
lc_eor = p21c.run_lightcone(
    lightconer=lightconer_eor,
    inputs=inputs_eor,
    write=CacheConfig.off(),
    include_dvdr_in_tau21=False,
)

# Fields are in lc.lightcones dict
print(f"  Available fields: {list(lc_eor.lightcones.keys())}")
print(f"  Lightcone shape: {lc_eor.lightcones['density'].shape}")
print(f"  Redshift range: {lc_eor.lightcone_redshifts.min():.2f} – {lc_eor.lightcone_redshifts.max():.2f}")

EoR lightconer: z = 6.00 – 10.00
  N slices = 518

Running EoR lightcone...


/Users/jelte/Documents/PhD/noise_analysis/.venv/lib/python3.11/site-packages/py21cmfast/drivers/coeval.py:898: UserWarning: You have turned off caching for the perturbed halo fields, but are evolving them across multiple redshifts. This will result in very high memory usage
  halofield_list = evolve_halos(
/Users/jelte/Documents/PhD/noise_analysis/.venv/lib/python3.11/site-packages/py21cmfast/wrapper/outputs.py:286: UserWarning: Trying to purge array 'halo_masses' from memory that hasn't been stored! Use force=True if you meant to do this.
  self._remove_array(k, force=force)
/Users/jelte/Documents/PhD/noise_analysis/.venv/lib/python3.11/site-packages/py21cmfast/wrapper/outputs.py:286: UserWarning: Trying to purge array 'star_rng' from memory that hasn't been stored! Use force=True if you meant to do this.
  self._remove_array(k, force=force)
/Users/jelte/Documents/PhD/noise_analysis/.venv/lib/python3.11/site-packages/py21cmfast/wrapper/outputs.py:286: UserWarning: Trying to purge arra

  Available fields: ['neutral_fraction', 'density', 'velocity_z']
  Lightcone shape: (128, 128, 518)
  Redshift range: 6.00 – 10.00


: 

In [ ]:
# ── Late-time lightcone (z=1 to 5) ──

lightconer_late = p21c.RectilinearLightconer.between_redshifts(
    min_redshift=Z_LATE_MIN,
    max_redshift=Z_LATE_MAX,
    resolution=resolution,
    quantities=lc_quantities,
)

print(f"Late-time lightconer: z = {lightconer_late.lc_redshifts.min():.2f} – {lightconer_late.lc_redshifts.max():.2f}")
print(f"  N slices = {len(lightconer_late.lc_redshifts)}")

print("\nRunning late-time lightcone...")
lc_late = p21c.run_lightcone(
    lightconer=lightconer_late,
    inputs=inputs_late,
    write=CacheConfig.off(),
    include_dvdr_in_tau21=False,
)

print(f"  Lightcone shape: {lc_late.lightcones['density'].shape}")
print(f"  Redshift range: {lc_late.lightcone_redshifts.min():.2f} – {lc_late.lightcone_redshifts.max():.2f}")
print(f"  Mean xHI: {lc_late.lightcones['neutral_fraction'].mean():.6f} (should be ~0 = fully ionized)")

Late-time lightconer: z = 1.00 – 5.00
  N slices = 1943

Running late-time lightcone...


In [ ]:
# ── Compute kSZ from EoR lightcone using our simple method ──

ksz_eor_simple = compute_ksz_map_lightcone_simple(
    lc_eor.lightcones["velocity_z"], lc_eor.lightcones["neutral_fraction"],
    lc_eor.lightcones["density"], lc_eor.lightcone_redshifts, BOX_LEN
)
print(f"EoR kSZ (lightcone, simple): std = {ksz_eor_simple.std():.4f} µK")

# Late-time lightcone skipped for now

In [ ]:
# ── Visual comparison: coeval vs EoR lightcone ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (m, label) in zip(axes, [
    (ksz_coeval, f"Coeval kSZ (z={Z_COEVAL})"),
    (ksz_eor_simple, f"EoR lightcone kSZ (z={Z_EOR_MIN}–{Z_EOR_MAX})"),
]):
    vmax = np.percentile(np.abs(m), 99)
    im = ax.imshow(m, cmap="RdBu_r", origin="lower",
                   extent=[0, BOX_LEN, 0, BOX_LEN], vmin=-vmax, vmax=vmax)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Mpc"); ax.set_ylabel("Mpc")
    plt.colorbar(im, ax=ax, label=r"$\Delta T_{\rm kSZ}$ [$\mu$K]")

plt.tight_layout()
plt.show()

## Part 3: wrapper.py kSZ (with exp(-τ), ray-tracing, rotation)

Now compute kSZ using the more complete `run_kSZ` from `ksz_reconstruction/wrapper.py` for the
EoR lightcone. This includes:
- Cumulative optical depth damping exp(-τ)
- Ray-tracing with angular diameter distance corrections
- Box rotation every HII_DIM slices to avoid repetition artifacts

We compare the wrapper result against our simple method on the same lightcone.

In [ ]:
# ── Compute EoR kSZ using wrapper.py _Proj_array directly ──
from wrapper import _Proj_array, _KszConstants, compute_tau
from astropy.cosmology import Planck18 as cosmo_planck

def compute_ksz_wrapper_method(lc_obj, z_start, seed=SEED):
    """Compute kSZ using the wrapper.py method. Returns kSZ map in µK."""
    h = cosmo_planck.H0.value / 100
    omb = cosmo_planck.Ob0
    kSZ_consts = _KszConstants(
        HII_DIM=HII_DIM, BOX_LEN=BOX_LEN, hlittle=h, OMb=omb,
        red_dist=len(lc_obj.lightcone_redshifts), redshift_start=z_start,
        DA_zstart=cosmo_planck.comoving_distance(z_start), Y_He=0.245,
    )
    kSZ_consts.mean_taue_curr_z = compute_tau(
        redshifts=[z_start], global_xHI=[1],
        user_params={"HII_DIM": HII_DIM, "BOX_LEN": BOX_LEN},
        cosmo_params={"hlittle": h, "OMb": omb},
    )
    import random; random.seed(seed)
    Tcmb, _ = _Proj_array(
        lc_obj.lightcone_redshifts,
        lc_obj.lightcones["density"],
        lc_obj.lightcones["velocity_z"],
        lc_obj.lightcones["neutral_fraction"],
        kSZ_consts, PARALLEL_APPROX=False, rotation=True,
    )
    ksz_uK = Tcmb * kSZ_consts.CMperMPC / constants.c.cgs.value * cosmo_planck.Tcmb0.value * 1e6
    return ksz_uK.astype(np.float32)

print("Computing EoR kSZ using wrapper method...")
ksz_eor_wrapper = compute_ksz_wrapper_method(lc_eor, Z_EOR_MIN)
print(f"EoR kSZ (wrapper): std = {ksz_eor_wrapper.std():.4f} µK")

# Late-time wrapper skipped for now

### wrapper.py kSZ method (from `ksz_reconstruction`)

The `run_kSZ()` function from `wrapper.py` implements a more complete kSZ calculation:

| Feature | Our simple method | wrapper.py method |
|---|---|---|
| Optical depth | Ignored (no exp(-τ) damping) | Cumulative τ with exp(-τ) attenuation |
| Electron density | `n_e = ρ_b(1+X_H)/2m_p` | `N_b0 = ρ_b/m_p × (1 - 0.75×Y_He)` + He contribution |
| Redshift factors | `n_e(z) × dl(z)` per slice | `dτ × (1+z)²` and `v × (1+z)` proper weighting |
| Ray-tracing | Parallel rays only | Angular diameter distance corrections |
| Box repetition | None | Random roll every HII_DIM slices |
| Mean removal | Not applied | Subtracts mean of Tcmb at end |

Both EoR and late-time lightcones need the wrapper method since both integrate
many slices along the LOS. The exp(-τ) damping matters more for deep lightcones.

### Compare all maps: coeval, EoR lightcone (simple + wrapper), late-time lightcone

In [ ]:
# ── Visual comparison: coeval vs EoR lightcone (simple + wrapper) ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

maps_labels = [
    (ksz_coeval, f"Coeval (z={Z_COEVAL}, simple)"),
    (ksz_eor_simple, "EoR LC (simple)"),
    (ksz_eor_wrapper, "EoR LC (wrapper.py)"),
]

for ax, (m, label) in zip(axes, maps_labels):
    vmax = np.percentile(np.abs(m), 99)
    im = ax.imshow(m, cmap="RdBu_r", origin="lower",
                   extent=[0, BOX_LEN, 0, BOX_LEN], vmin=-vmax, vmax=vmax)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Mpc"); ax.set_ylabel("Mpc")
    plt.colorbar(im, ax=ax, label=r"$\Delta T_{\rm kSZ}$ [$\mu$K]", shrink=0.9)

plt.suptitle("kSZ maps: coeval vs EoR lightcone, simple vs wrapper", fontsize=13)
plt.tight_layout()
plt.show()

## Part 4: Full non-Gaussianity comparison

Compare non-Gaussianity statistics, PDFs, and Minkowski functionals for all maps:
1. Coeval kSZ (z=8, single snapshot)
2. EoR lightcone kSZ — simple method (z=5–12)
3. EoR lightcone kSZ — wrapper method (z=5–12)
4. Late-time lightcone kSZ — simple method (z=0.5–5)
5. Late-time lightcone kSZ — wrapper method (z=0.5–5)

**Key question:** Is the EoR kSZ (from bubbles) significantly more non-Gaussian than the
late-time kSZ (from linear density/velocity)? Does the lightcone integration wash this out?

In [ ]:
# ── Pixel statistics for available maps ──
all_maps = {
    f"Coeval (z={Z_COEVAL})": ksz_coeval,
    "EoR LC (simple)": ksz_eor_simple,
    "EoR LC (wrapper)": ksz_eor_wrapper,
    # Late-time maps skipped for now
}

all_stats = {}
for label, m in all_maps.items():
    all_stats[label] = pixel_statistics(m, label=label)
    print()

In [ ]:
# ── Summary bar chart of non-Gaussianity metrics ──
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

labels = list(all_stats.keys())
short_labels = [l.replace("(", "\n(") for l in labels]
colors = ["C0", "C2", "C4"]

metrics = [
    ("skewness", "Skewness (Gauss=0)"),
    ("excess_kurtosis", "Excess Kurtosis (Gauss=0)"),
    ("m5 (5th moment)", r"$m_5$ (Gauss=0)"),
    ("m6_excess (6th moment - 15)", r"$m_6 - 15$ (Gauss=0)"),
]

for ax, (key, title) in zip(axes, metrics):
    vals = [all_stats[l][key] for l in labels]
    bars = ax.bar(short_labels, vals, color=colors, edgecolor="k", alpha=0.8)
    ax.axhline(0, color="k", ls="--", lw=0.8)
    ax.set_title(title, fontsize=10)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.tick_params(axis="x", labelsize=8)

plt.suptitle("Non-Gaussianity: higher-order moments", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pixel PDF for all maps ──
fig = plot_pixel_pdf(all_maps)
plt.show()

In [ ]:
# ── Minkowski functionals for available maps ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

mf_colors = list(zip(all_maps.keys(), ["C0", "C2", "C4"]))

for (label, m), (_, color) in zip(all_maps.items(), mf_colors):
    nu, V0, V1, V2, _, _, _ = minkowski_functionals_2d(m)
    axes[0].plot(nu, V0, color=color, label=label, lw=1.5)
    axes[1].plot(nu, V1, color=color, label=label, lw=1.5)
    axes[2].plot(nu, V2, color=color, label=label, lw=1.5)

# Gaussian reference
gauss_map = np.random.randn(HII_DIM, HII_DIM).astype(np.float32)
nu_g, V0g, V1g, V2g, _, _, _ = minkowski_functionals_2d(gauss_map)
axes[0].plot(nu_g, V0g, "k:", lw=1.5, alpha=0.7, label="Gaussian")
axes[1].plot(nu_g, V1g, "k:", lw=1.5, alpha=0.7, label="Gaussian")
axes[2].plot(nu_g, V2g, "k:", lw=1.5, alpha=0.7, label="Gaussian")

titles = [r"$V_0$ (area fraction)", r"$V_1$ (perimeter)", r"$V_2$ (Euler char.)"]
for ax, title in zip(axes, titles):
    ax.set_xlabel(r"$\nu = (T - \bar{T})/\sigma$")
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle("Minkowski Functionals — kSZ maps vs Gaussian", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Part 5: Power spectrum comparison — simple vs wrapper method

Compare the 2D angular power spectra of kSZ maps from our simple method vs the wrapper.py
method to quantify where the methods differ in Fourier space.

In [ ]:
# ── Power spectra of available kSZ maps ──
fig, ax = plt.subplots(figsize=(9, 6))

ps_specs = [
    (f"Coeval (z={Z_COEVAL})", ksz_coeval, "C0", "-"),
    ("EoR LC (simple)", ksz_eor_simple, "C2", "-"),
    ("EoR LC (wrapper)", ksz_eor_wrapper, "C4", "--"),
]

for label, m, color, ls in ps_specs:
    m_zero = (m - m.mean()).astype(np.float64)
    P, k = get_power(m_zero, boxlength=[BOX_LEN, BOX_LEN], bins=30,
                     ignore_zero_mode=True, dimensionless=False)
    ax.loglog(k, P * k**2 / (2 * np.pi), color=color, ls=ls, lw=2, label=label)

ax.set_xlabel(r"$k$ [Mpc$^{-1}$]")
ax.set_ylabel(r"$\Delta^2_{\rm kSZ}(k) = k^2 P(k) / 2\pi$ [$\mu$K$^2$]")
ax.set_title("kSZ power spectra: coeval vs EoR lightcone")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cross-correlation: simple vs wrapper EoR kSZ ──

def cross_correlate_2d(map1, map2, box_len, bins=50):
    """Fourier-space cross-correlation coefficient r(k)."""
    f1 = (map1 - map1.mean()).astype(np.float64)
    f2 = (map2 - map2.mean()).astype(np.float64)
    bl = [box_len, box_len]
    kw = dict(boxlength=bl, bins=bins, ignore_zero_mode=True, dimensionless=False)
    P_11, k = get_power(deltax=f1, **kw)
    P_22 = get_power(deltax=f2, **kw)[0]
    P_12 = get_power(deltax=f1, deltax2=f2, **kw)[0]
    denom = np.sqrt(np.abs(P_11 * P_22))
    r_k = np.where(denom > 0, P_12 / denom, 0.0)
    r_real = float(np.sum(f1.ravel() * f2.ravel()) /
                   np.sqrt(np.sum(f1.ravel()**2) * np.sum(f2.ravel()**2)))
    return k, r_k, r_real

k1, rk1, r1 = cross_correlate_2d(ksz_eor_simple, ksz_eor_wrapper, BOX_LEN)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(k1, rk1, "C2", lw=2)
ax.set_title(f"EoR: simple vs wrapper (Pearson r = {r1:.3f})")
ax.set_xlabel(r"$k$ [Mpc$^{-1}$]"); ax.set_ylabel(r"$r(k)$")
ax.axhline(1.0, color="k", ls="--", lw=0.8)
ax.set_ylim(-0.5, 1.1); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 6: Fourier-space non-Gaussianity — bispectrum estimation

The **bispectrum** (3-point function in Fourier space) is the leading-order non-Gaussian statistic.
The trispectrum (4-point) is what current methods use to separate reionization vs late-time kSZ.
Here we estimate the angle-averaged bispectrum for equilateral triangles as a function of k,
to see if the reionization kSZ has a significantly stronger bispectrum than the late-time signal.

In [ ]:
def estimate_equilateral_bispectrum(map_2d, box_len, n_k_bins=15, dk_frac=0.2):
    """Estimate the equilateral bispectrum B(k,k,k) for a 2D map.
    
    For each k bin, find all Fourier mode triplets (k1, k2, k3) with |ki| ≈ k
    that form a closed triangle k1 + k2 + k3 = 0, and average their product.
    
    This is a simplified estimator using the binned approach.
    """
    ny, nx = map_2d.shape
    f = np.fft.fft2(map_2d - map_2d.mean()) / (nx * ny)
    
    dk = 2 * np.pi / box_len
    kx = np.fft.fftfreq(nx, d=1.0/nx) * dk
    ky = np.fft.fftfreq(ny, d=1.0/ny) * dk
    KX, KY = np.meshgrid(kx, ky, indexing="ij")
    K = np.sqrt(KX**2 + KY**2)
    
    k_min = dk
    k_max = min(nx, ny) // 2 * dk * 0.8
    k_edges = np.linspace(k_min, k_max, n_k_bins + 1)
    k_centers = 0.5 * (k_edges[:-1] + k_edges[1:])
    
    B_eq = np.zeros(n_k_bins)
    B_counts = np.zeros(n_k_bins)
    
    for ib in range(n_k_bins):
        klo, khi = k_edges[ib], k_edges[ib + 1]
        # Modes in this bin
        mask = (K >= klo) & (K < khi)
        idx = np.argwhere(mask)
        
        if len(idx) < 3:
            continue
        
        # For equilateral: pick pairs and check if third leg closes
        # Use a Monte Carlo subset for efficiency
        n_sample = min(len(idx), 200)
        rng = np.random.default_rng(42)
        subset = idx[rng.choice(len(idx), n_sample, replace=False)]
        
        count = 0
        bsum = 0.0
        for i in range(n_sample):
            ix1, iy1 = subset[i]
            for j in range(i + 1, n_sample):
                ix2, iy2 = subset[j]
                # Third mode must close the triangle: k3 = -(k1 + k2)
                ix3 = (-ix1 - ix2) % nx
                iy3 = (-iy1 - iy2) % ny
                k3_mag = K[ix3, iy3]
                if klo <= k3_mag < khi:
                    bsum += np.real(f[ix1, iy1] * f[ix2, iy2] * f[ix3, iy3])
                    count += 1
        
        if count > 0:
            B_eq[ib] = bsum / count
            B_counts[ib] = count
    
    return k_centers, B_eq, B_counts


print("Estimating equilateral bispectrum (this may take ~1 min per map)...")
bispec_results = {}
for label, m in all_maps.items():
    print(f"  {label}...", end=" ", flush=True)
    k_b, B_b, counts = estimate_equilateral_bispectrum(m, BOX_LEN)
    bispec_results[label] = (k_b, B_b, counts)
    print("done")

print("Bispectrum estimation complete.")

In [ ]:
# ── Plot equilateral bispectrum ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_list = ["C0", "C1", "C2", "C3"]
for (label, (k_b, B_b, counts)), color in zip(bispec_results.items(), colors_list):
    valid = counts > 10  # only plot bins with enough triangles
    # Left: raw bispectrum
    axes[0].plot(k_b[valid], np.abs(B_b[valid]), "o-", color=color, label=label, ms=4)
    # Right: reduced bispectrum b = B / (P1*P2 + P2*P3 + P1*P3)
    # Normalize by power spectrum for shape comparison
    m = list(all_maps.values())[list(all_maps.keys()).index(label)]
    m_z = (m - m.mean()).astype(np.float64)
    P, k_p = get_power(m_z, boxlength=[BOX_LEN, BOX_LEN], bins=len(k_b),
                        ignore_zero_mode=True, dimensionless=False)
    # Interpolate P to bispectrum k bins
    P_interp = np.interp(k_b, k_p, P, left=P[0], right=P[-1])
    denom = 3 * P_interp**2  # equilateral: 3 * P(k)^2
    b_reduced = np.where(denom > 0, B_b / denom, 0)
    axes[1].plot(k_b[valid], b_reduced[valid], "o-", color=color, label=label, ms=4)

axes[0].set_xlabel(r"$k$ [Mpc$^{-1}$]")
axes[0].set_ylabel(r"$|B_{\rm eq}(k)|$ [$\mu$K$^3$ Mpc$^4$]")
axes[0].set_title("Equilateral bispectrum (absolute)")
axes[0].set_yscale("log")
axes[0].set_xscale("log")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel(r"$k$ [Mpc$^{-1}$]")
axes[1].set_ylabel(r"$b_{\rm eq}(k) = B / 3P^2$")
axes[1].set_title("Reduced bispectrum (shape)")
axes[1].set_xscale("log")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0, color="k", ls="--", lw=0.8)

plt.suptitle("Bispectrum — reionization vs late-time kSZ", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Part 7: ML separability test

TODO: Once late-time lightcone is enabled, train a CNN to distinguish
EoR from late-time kSZ patches. For now, compare coeval vs EoR lightcone
to check if the CNN can detect the difference in non-Gaussianity between
a single-z snapshot and a LOS-integrated signal.

In [ ]:
# ── Skipped for now — requires late-time lightcone ──
# Re-enable when lc_late is available.
# For now, the non-Gaussianity comparison between coeval and EoR lightcone
# (Parts 4-6 above) already shows the key results.
print("ML training skipped — late-time lightcone not yet computed.")

In [ ]:
# Skipped — see cell above

In [ ]:
# Skipped — see cell above

## Part 8: Controlled ML test — power spectrum equalized

Skipped for now — requires late-time lightcone.

In [ ]:
# Skipped — see cell above

In [ ]:
# Skipped — see cell above

## Summary

### Non-Gaussianity diagnostics
- **Pixel statistics**: Skewness, excess kurtosis, and higher moments (m5, m6) quantify deviations from Gaussianity. The EoR kSZ should show significantly larger values than late-time kSZ due to bubble morphology.
- **Minkowski functionals**: V0, V1, V2 as functions of threshold are sensitive to topology. Ionized bubbles in EoR kSZ create characteristic deviations from Gaussian MFs.
- **Bispectrum**: The equilateral bispectrum B(k,k,k) and reduced form b(k) directly measure 3-point non-Gaussianity.

### EoR vs Late-time kSZ (the key comparison)
- **EoR lightcone** (z=5–12): Full LOS-integrated reionization kSZ from patchy ionization bubbles. Non-Gaussianity comes from the binary-like structure of x_e (ionized/neutral).
- **Late-time lightcone** (z=0.5–5): Fully ionized universe, kSZ from density/velocity bulk flows only. Should be closer to Gaussian since density/velocity are nearly Gaussian at these scales.
- Both use proper lightcone integration (not single coeval snapshots) for realistic signals.

### Coeval vs Lightcone
- **Coeval** (z=8): Single snapshot captures peak bubble structure — maximum non-Gaussianity.
- **Lightcone** (z=5–12): Integrates over evolving reionization — CLT may reduce non-Gaussianity but bubble signal should persist since it's correlated along the LOS.

### Our method vs wrapper.py
| Feature | Our method | wrapper.py |
|---|---|---|
| Optical depth | No exp(-τ) | Cumulative exp(-τ) damping |
| Electron density | Simple n_e = ρ_b(1+X_H)/2m_p | Full N_b0 with He: (1-0.75Y_He) + He/4 |
| (1+z) factors | Per-slice pref(z) | Explicit (1+z)² on dτ and (1+z) on v |
| Ray-tracing | Parallel rays only | Angular diameter distance corrections |
| Box repetition | None | Random roll every HII_DIM slices |

### ML separability
- **Original CNN**: Tests overall separability (power spectrum + non-Gaussianity) between EoR and late-time lightcone kSZ.
- **PS-equalized CNN**: Tests whether beyond-2-point information alone can separate the signals. If yes, this supports the idea that ML could outperform trispectrum-based methods by exploiting higher-order correlations from the bubble morphology of patchy reionization.